# Statistical Inference of Typhoon Impacts and Infrastructure Resilience

In this notebook, we perform formal statistical inference to test the validity of our hypotheses regarding the relationship between meteorological intensity, infrastructure investment, and socio-economic outcomes.

The purpose of this notebook is to move beyond exploratory observation and confirm whether the patterns identified in our datasets are statistically significant or merely the result of random chance. This is achieved by applying classical statistical tests to our integrated dataset, which combines typhoon observations with cumulative flood control investment metrics.

The analysis includes:

- Independent T-Tests: To determine if infrastructure planning is responsive to meteorological risks like high-intensity rainfall.

- Chi-Square Tests of Independence: To examine the relationship between "Fiscal Friction" (budgetary inefficiency) and typhoon mortality rates.

- Two-Sample Z-Tests for Proportions: To assess the "Investment Success" of flood control projects and the "False Security" provided during extreme disaster events.

After completing these tests, we will have a mathematically grounded basis to confirm or disprove our conclusions regarding the efficacy of flood control systems and the socio-political factors influencing disaster resilience in the Philippines.


### Import

Start by importing **pandas**, **numpy**, **scipy**, and **statsmodel**.


In [7]:
# Import relevant python modules
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

## Loading and Merging the Complete Datasets

In this step, we processed datasets by merging the meteorological, infrastructure context, and human impact datasets into a single, unified DataFrame (`df_final`). Because our statistical tests rely on evaluating relationships across different domains—such as comparing infrastructure budget variance with typhoon mortality—all relevant variables must exist within the same table.

The following DataFrames are initialized to facilitate the analysis:

- `df_meteo_infra`: Contains meteorological observations (rainfall, wind speed) enriched with cumulative flood control investment context per province.

- `df_infra`: Contains the individual flood control project details, including the approved budgets and actual contract costs needed for financial accuracy testing.

- `df_impacts`: Contains the standardized human and structural impact records, such as the number of affected persons and casualties.


In [8]:
# 1. Load the meteorological + infrastructure  dataset
df_meteo_infra = pd.read_csv('../data/merged/typhoon-info-infra-project.csv')

# 2. Load the raw infrastructure projects 
df_infra = pd.read_csv('../data/infra-projects/cleaned_infra_projects.csv')

# 3. Load the impacts dataset (contains Deaths, Affected, Damage, Category)
df_impacts = pd.read_csv('../data/merged/cleaned_typhoon_impacts.csv')

# Rename 'Cyclone Name' to match the 'Typhoon' column in df_meteo_infra
df_impacts = df_impacts.rename(columns={'Cyclone Name': 'Typhoon'})

# Merge them together into df_final based on storm, year, and region
df_final = pd.merge(df_meteo_infra, df_impacts, on=['Typhoon', 'Year', 'Region'], how='inner')

## Step 2: Testing of Infrastructure Planning vs. Rainfall Intensity

In this step, we perform an Independent Two-Sample T-Test to determine if infrastructure planning is adequately responsive to meteorological risks.

**Preprocessing & Threshold Justification:** We categorize provinces into two distinct groups: "High Intensity Zones" and "Low Intensity Zones." A province is designated as high-intensity if it frequently experiences rainfall `exceeding 150 mm`. By analyzing the continuous variable `Final_Budget_M` across these two groups, we can statistically verify whether the government allocates significantly more funds to objectively higher-risk areas, or if the budget distribution is driven by other socio-political factors.

**Hypotheses:**

- Null Hypothesis ($H_0$): The mean budget allocation in high-intensity rainfall zones is the same as or lower than the mean budget allocation in low-intensity zones ($\mu_{high} \le \mu_{low}$).
- Alternative Hypothesis ($H_a$): The mean budget allocation in high-intensity rainfall zones is significantly higher than in low-intensity zones ($\mu_{high} > \mu_{low}$).

**Assumptions & Requirements:**

- **Independence:** The budget allocations and rainfall patterns for each province are independent of one another.
- **Normality:** The budget allocations (`Final_Budget_M`) within both high and low-intensity zones should be approximately normally distributed.
- **Unequal Variances:** We assume unequal variances in budget allocations across both groups, which justifies using Welch's T-test via `equal_var=False`
- **Significance Level:** $\alpha = 0.05$.


In [9]:
# Grouping by province to find frequency of high-intensity rain (>150mm)
intensity_counts = df_meteo_infra.groupby('Province')['Max 24-hour Rainfall (mm)'].apply(lambda x: (x > 150).sum())
median_freq = intensity_counts.median()

high_zone_provinces = intensity_counts[intensity_counts > median_freq].index
low_zone_provinces = intensity_counts[intensity_counts <= median_freq].index

# Compare Final_Budget_M for projects in these zones using the raw infrastructure data
high_zone_budgets = df_infra[df_infra['Province'].isin(high_zone_provinces)]['Final_Budget_M']
low_zone_budgets = df_infra[df_infra['Province'].isin(low_zone_provinces)]['Final_Budget_M']

# Using the T-test (equal_var=False) for better accuracy with budget data
t_stat_rain, p_val_rain = stats.ttest_ind(high_zone_budgets, low_zone_budgets, nan_policy='omit', equal_var=False, alternative='greater')

# Print the averages 
print(f"High Intensity Zone Average Budget: {high_zone_budgets.mean():.2f} Million")
print(f"Low Intensity Zone Average Budget: {low_zone_budgets.mean():.2f} Million")
print(f"Rainfall vs Planning P-Value: {p_val_rain:.4f}")

High Intensity Zone Average Budget: 51.54 Million
Low Intensity Zone Average Budget: 58.00 Million
Rainfall vs Planning P-Value: 1.0000


### Interpretation: Fail to Reject the Null Hypothesis ($H_0$)

The Independent T-test yielded a p-value of **1.0000e+00**. Because this value is far above the standard significance level of 0.05, we **fail to reject the null hypothesis ($H_0$)**.

There is no statistical evidence to support the claim that high-intensity rainfall zones receive significantly higher budget allocations. Because the directional p-value is exactly 1.0, the data actually strongly suggests the opposite: drier, low-intensity regions are receiving significantly more funding than the most meteorologically at-risk provinces. This shows a major disconnect between objective climate risk and infrastructure planning, proving that flood control budgets in the Philippines are not distributed based on rainfall vulnerability. Instead, it suggests that other non-meteorological variables, such as economic value, population density, or political priorities, are the primary drivers of fund allocation.


# Step 3: Testing "Fiscal Friction" and Mortality

In this step, we use a Chi-Square Test of Independence to examine the relationship between financial inefficiencies and human casualties during typhoon events.

**Preprocessing & Threshold Justification:** To construct our contingency table, we classify our data into two categorical variables. The first variable, "Fiscal Friction," is defined as "High Friction" if the variance ratio between the approved budget and the actual contract cost exceeds 10%, indicating significant budgetary gaps or project delays. The second variable categorizes disaster outcomes into "High Mortality" and "Low/Zero Mortality" based on the occurrence of casualties. This binary categorization allows us to statistically evaluate whether project execution inefficiencies are significantly associated with deadlier disaster outcomes.

**Hypotheses:**

- Null Hypothesis ($H_0$): Fiscal friction and mortality rates are independent; the presence of high financial inefficiency does not alter the likelihood of higher casualties.
- Alternative Hypothesis ($H_a$): There is a significant association between high fiscal friction and increased mortality rates .

**Assumptions & Requirements:**

- **Categorical Data:** Both variables being tested (Fiscal Friction status and Mortality category) must be categorical.
- **Independence of Observations:** Each recorded event or project area must represent an independent observation; one event's outcome should not influence another's.
- **Expected Cell Counts:** The expected frequency in each cell of the $2 \times 2$ contingency table must be at least 5.
- **Significance Level:** $\alpha = 0.05$.


In [10]:
df_final['High_Friction'] = df_final['Variance_Ratio_To_Date'] > 0.10
df_final['High_Mortality'] = df_final['Deaths'] > 0

# Create a 2x2 contingency table
contingency = pd.crosstab(df_final['High_Friction'], df_final['High_Mortality'])
chi2, p_val_friction, dof, ex = stats.chi2_contingency(contingency)

print(f"Fiscal Friction vs Mortality P-Value: {p_val_friction:.4f}")

Fiscal Friction vs Mortality P-Value: 0.6686


### Interpretation: Fail to Reject the Null Hypothesis ($H_0$)

The Chi-Square Test of Independence yielded a p-value of **0.6686**. Because this value is well above our significance level of 0.05, we **fail to reject the null hypothesis ($H_0$)**.

There is no significant evidence in this dataset to suggest that "Fiscal Friction" (having a budget variance greater than 10%) is associated with higher mortality rates during typhoons. This means we cannot definitively conclude that financial inefficiencies or budgetary gaps directly lead to deadlier disaster outcomes. It suggests that while fiscal friction might delay projects or waste money, it does not necessarily strip the infrastructure of its baseline ability to save lives. It also implies that typhoon casualties are likely driven by other complex factors—such as the sheer severity of the storm, geographical vulnerability, or the presence of early warning evacuation systems—rather than just the financial efficiency of the flood control projects alone.


## Step 4: Testing "False Security" - Infrastructure Efficacy vs. Typhoon Intensity

In this step, we use a Two-Sample Z-Test for Proportions to evaluate if high-budget flood control projects provide the same level of absolute protection during Super Typhoons as they do during regular typhoons.

**Preprocessing & Threshold Justification:**
We define "Protection Success" as a zero-casualty event (`Deaths == 0`). We isolate only the regions with "High Budget" infrastructure (using the median `Cumulative_Budget_To_Date` to create a balanced, data-driven threshold). By isolating the top 50% of funded areas, we can statistically verify if extreme meteorological events (categorized as "Super Typhoons") overwhelm even the most well-funded infrastructure compared to standard typhoons.

**Hypotheses:**

- Null Hypothesis ($H_0$): The proportion of zero-casualty events in high-budget areas is the same or higher for Super Typhoons compared to regular typhoons.
- Alternative Hypothesis ($H_a$): The proportion of zero-casualty events in high-budget areas is significantly lower for Super Typhoons compared to regular typhoons.

**Assumptions & Requirements:**

- **Independence:** We assume that typhoon events and casualty outcomes are independent observations.
- **Large Sample Size:** We assume our historical dataset provides a sufficiently large sample size where the expected number of successes and failures in each group is at least 5 for the normal approximation to be valid.
- **Significance Level:** $\alpha = 0.05$.


In [11]:
# Preprocessing: Isolate high-budget areas 
median_budget = df_final['Cumulative_Budget_To_Date'].median()
high_budget_df = df_final[df_final['Cumulative_Budget_To_Date'] > median_budget].copy()

# Preprocessing: Identify Super Typhoons 
high_budget_df['Is_Super'] = high_budget_df['Category'].astype(str).str.contains('Super', case=False, na=False)

# Define "Success" as zero deaths
success_super = high_budget_df[high_budget_df['Is_Super']]['Deaths'] == 0
success_reg = high_budget_df[~high_budget_df['Is_Super']]['Deaths'] == 0

# Set up counts and number of observations for the Z-test
count_intensity = np.array([success_super.sum(), success_reg.sum()])
nobs_intensity = np.array([len(success_super), len(success_reg)])

# Perform Two-Sample Z-Test for Proportions (Alternative = 'smaller')
stat_intensity, p_val_intensity = proportions_ztest(count_intensity, nobs_intensity, alternative='smaller')

# Print success rates for transparency
print(f"Super Typhoon Success Rate: {count_intensity[0]/nobs_intensity[0]:.2%}")
print(f"Regular Typhoon Success Rate: {count_intensity[1]/nobs_intensity[1]:.2%}")
print(f"Efficacy vs Intensity P-Value: {p_val_intensity:.4f}")

Super Typhoon Success Rate: 56.25%
Regular Typhoon Success Rate: 73.13%
Efficacy vs Intensity P-Value: 0.0465


### Interpretation: Reject the Null Hypothesis

The Two-Sample Z-Test for Proportions yielded a p-value of **0.0465**. Because this value is just below the standard significance level of 0.05, **we reject the null hypothesis ($H_0$)**.

This significant result indicates that high-budget infrastructure does not offer the same level of protection during Super Typhoons as it does for regular typhoons. The sample data shows a measurable drop in "zero-casualty" success rates (from 73.13% down to 56.25%) when a Super Typhoon hits. This proves that extreme meteorological events can still overwhelm even heavily funded flood control projects, exposing a "false security" if residents and planners rely solely on infrastructure volume for safety during extreme weather events.


## Step 5: Testing "Investment Success" - Affected Populations vs. Infrastructure Volume

In this final step, we use a Two-Sample Z-Test for Proportions to determine if a higher volume of infrastructure investment actually protects a higher percentage of the population from being displaced or affected.

**Preprocessing & Threshold Justification:**
We categorize regions into "High Budget" and "Low Budget" groups based on the median `Cumulative_Budget_To_Date`. Using the median ensures a balanced, data-driven binary split. A "High Protection" event is defined as one where the number of `Affected` individuals is strictly below the overall dataset median.

**Hypotheses:**

- Null Hypothesis ($H_0$): The proportion of high-protection events is the same or lower in high-budget areas compared to low-budget areas.
- Alternative Hypothesis ($H_a$): The proportion of high-protection events is significantly higher in high-budget areas compared to low-budget areas.

**Assumptions & Requirements:**

- **Independence:** The flood events and budget impacts are assumed to be independent across different regions.
- **Large Sample Size:** We assume our historical dataset provides a sufficiently large sample size where the expected number of successes and failures in each group is at least 5 for the normal approximation to be valid.
- **Significance Level:** $\alpha = 0.05$.


In [12]:
# Preprocessing: Define High Budget regions
median_overall_budget = df_final['Cumulative_Budget_To_Date'].median()
df_final['High_Budget'] = df_final['Cumulative_Budget_To_Date'] > median_overall_budget

# Preprocessing: Define "High Protection" as having fewer affected people than the overall median
median_affected = df_final['Affected'].median()
df_final['High_Protection'] = df_final['Affected'] < median_affected

# Extract successes (High Protection = True) for both budget groups
protect_high_bud = df_final[df_final['High_Budget']]['High_Protection']
protect_low_bud = df_final[~df_final['High_Budget']]['High_Protection']

# Set up counts and number of observations for the Z-test
count_protect = np.array([protect_high_bud.sum(), protect_low_bud.sum()])
nobs_protect = np.array([len(protect_high_bud), len(protect_low_bud)])

# Perform Two-Sample Z-Test for Proportions (Alternative = 'larger')
stat_protect, p_val_protect = proportions_ztest(count_protect, nobs_protect, alternative='larger')

# Print Success Rates
print(f"High Budget Success Rate: {count_protect[0]/nobs_protect[0]:.2%}")
print(f"Low Budget Success Rate: {count_protect[1]/nobs_protect[1]:.2%}")
print(f"Affected vs Volume P-Value: {p_val_protect:.4f}")

High Budget Success Rate: 36.36%
Low Budget Success Rate: 54.69%
Affected vs Volume P-Value: 0.9990


The Two-Sample Z-Test for Proportions yielded a p-value of **0.999**. Because this value is well above the standard significance level of 0.05, **we fail to reject the null hypothesis ($H_0$)**.

There is no statistical evidence in this directional test to claim that high-budget areas experience a greater proportion of high-protection events compared to low-budget areas. The sample data actually shows a lower success rate for high-budget regions (36.36% vs 54.69%). While we cannot definitively prove the inverse relationship without further testing, this outcome implies that simply pouring more money into infrastructure volume does not guarantee fewer people will be affected. Disaster displacement is likely driven more by population density and geographic vulnerability in those high-budget areas than by the financial volume alone.
